# Cross-Study Shi Seurat Prediction Scores

Notebook front end for the Shi-reference Seurat label-transfer plotting workflow. The reusable implementation lives in `mge_organoid_python.cross_study_shi_prediction_plots`.

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "python_notebooks" / "src" / "mge_organoid_python").exists()),
    Path.cwd(),
)
SRC_DIR = REPO_ROOT / "python_notebooks" / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mge_organoid_python.cross_study_shi_prediction_plots import (
    RUN_LABEL_DEFAULT,
    combine_tables,
    load_combined_table,
    output_paths,
    plot_all,
    print_final_report,
    setup_tables,
)

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/nfs/turbo/umms-parent/mgeo_neuron_scrnaseq_projectfolder"))
RUN_LABEL = os.environ.get("CROSS_STUDY_SHI_RUN_LABEL", RUN_LABEL_DEFAULT)
MAX_CELLS_RAW = os.environ.get("CROSS_STUDY_SHI_MAX_CELLS_PER_STUDY", "").strip()
MAX_CELLS_PER_STUDY = int(MAX_CELLS_RAW) if MAX_CELLS_RAW else None

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"RUN_LABEL={RUN_LABEL}")
print(f"MAX_CELLS_PER_STUDY={MAX_CELLS_PER_STUDY}")

In [ ]:
ready = setup_tables(PROJECT_ROOT, RUN_LABEL)
ready

In [ ]:
missing = ready.loc[~ready["per_study_obs_exists"], ["study_id", "per_study_obs_path"]]
if not missing.empty:
    print("Per-study Shi prediction tables are not all present yet:")
    print(missing.to_string(index=False))
else:
    obs = combine_tables(PROJECT_ROOT, RUN_LABEL)
    print(obs[["study_id", "study_label"]].value_counts(sort=False).to_string())

In [ ]:
ready = setup_tables(PROJECT_ROOT, RUN_LABEL)
if ready["per_study_obs_exists"].all():
    result = plot_all(PROJECT_ROOT, RUN_LABEL, max_cells_per_study=MAX_CELLS_PER_STUDY)
    print_final_report(output_paths(PROJECT_ROOT, RUN_LABEL))
else:
    print("Skipping plots until the Seurat transfer/export step has produced every per-study obs table.")